In [11]:
import nltk
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re, collections

# Download necessary NLTK data (run once)
nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries imported and data downloaded.")

Libraries imported and data downloaded.


In [12]:
# Load 'shakespeare-caesar.txt' from Gutenberg corpus
file_id = 'shakespeare-caesar.txt'
raw_text = gutenberg.raw(file_id)

print(f"--- Loaded Corpus: {file_id} ---")
print(f"Total characters: {len(raw_text)}")
print(f"Sample text (first 100 chars): \n{raw_text[:100]}...")

--- Loaded Corpus: shakespeare-caesar.txt ---
Total characters: 112310
Sample text (first 100 chars): 
[The Tragedie of Julius Caesar by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Fla...


In [13]:
# Tokenize the text
ptb_tokens = word_tokenize(raw_text)

print("--- Penn Treebank Tokenization ---")
print(f"Total Tokens found: {len(ptb_tokens)}")
print(f"First 20 tokens: {ptb_tokens[:20]}")

--- Penn Treebank Tokenization ---
Total Tokens found: 25277
First 20 tokens: ['[', 'The', 'Tragedie', 'of', 'Julius', 'Caesar', 'by', 'William', 'Shakespeare', '1599', ']', 'Actus', 'Primus', '.', 'Scoena', 'Prima', '.', 'Enter', 'Flauius', ',']


In [14]:
# --- BPE Helper Functions ---
def get_stats(vocab):
    """Compute frequency of adjacent pairs of symbols."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    """Merge the most frequent pair into a new token."""
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# --- Execution ---
print("--- Byte Pair Encoding (BPE)  ---")

# 1. Create a sample corpus for BPE training
bpe_corpus_sample = "low lower newest widest"
print(f"Training Corpus: '{bpe_corpus_sample}'")

# 2. Pre-tokenization: Split by space and add end-of-word token </w>
vocab = collections.defaultdict(int)
for word in bpe_corpus_sample.split():
    # 'low' becomes 'l o w </w>'
    vocab[' '.join(list(word)) + ' </w>'] += 1

# 3. Perform Merges (5 Iterations)
num_merges = 5
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Iter {i+1}: Merged pair {best}")

print("\nFinal Vocabulary State:")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

--- Byte Pair Encoding (BPE)  ---
Training Corpus: 'low lower newest widest'
Iter 1: Merged pair ('l', 'o')
Iter 2: Merged pair ('lo', 'w')
Iter 3: Merged pair ('e', 's')
Iter 4: Merged pair ('es', 't')
Iter 5: Merged pair ('est', '</w>')

Final Vocabulary State:
low </w>: 1
low e r </w>: 1
n e w est</w>: 1
w i d est</w>: 1


In [15]:
stemmer = PorterStemmer()

# Select a sample of words to stem (e.g., words 500 to 520)
sample_tokens = ptb_tokens[500:520]
stemmed_tokens = [stemmer.stem(token) for token in sample_tokens]

print("--- Porter Stemmer Results ---")
print(f"{'Original':<15} | {'Stemmed':<15}")
print("-" * 33)

for orig, stem in zip(sample_tokens, stemmed_tokens):
    print(f"{orig:<15} | {stem:<15}")

--- Porter Stemmer Results ---
Original        | Stemmed        
---------------------------------
bankes          | bank           
To              | to             
heare           | hear           
the             | the            
replication     | replic         
of              | of             
your            | your           
sounds          | sound          
,               | ,              
Made            | made           
in              | in             
her             | her            
Concaue         | concau         
Shores          | shore          
?               | ?              
And             | and            
do              | do             
you             | you            
now             | now            
put             | put            
